# NB-02 — めぐ指数: 統合回帰モデル推定

**目的**: ペース・馬場・斤量・レースレベルの4補正係数（β₁〜β₄）と基準タイム（固定効果）を単一の OLS 回帰で一括推定する

**前提**: NB-01 の `megu_dataset.parquet` が生成済みであること

**出力**:
- `megu_regression_params` テーブル（β₁〜β₄ の推定値）
- `megu_par_time` テーブル（距離×コース×芝ダート×馬場カテゴリ別の基準タイム）

**回帰式**:
```
raw_time = β₀
         + β₁ × front_split_dev      # ペース補正係数
         + β₂ × TSI_offset           # 馬場補正係数
         + β₃ × weight_dev × dist_scale  # 斤量補正係数
         + β₄ × log(FQ / par_FQ)    # レースレベル補正係数
         + Σγᵢ × fixed_effect_i      # 距離×コース×芝ダート×馬場カテゴリ
         + ε
```

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/keiba-vpn')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from pathlib import Path

INPUT_DIR  = Path('output/nb01')
OUTPUT_DIR = Path('output/nb02')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(INPUT_DIR / 'megu_dataset.parquet')
print(f'読み込み完了: {len(df):,} 行')
print(df.dtypes)

In [ ]:
# ===== 学習・テスト期間設定 =====
# 本番データ使用時: 学習=2024-2025年、テスト=2026年以降
# モックデータ使用時も同じ変数を参照し、日付フィルタで自動的に適用される
TRAIN_START   = pd.Timestamp('2024-01-01')
TRAIN_END     = pd.Timestamp('2025-12-31')
TEST_START    = pd.Timestamp('2026-01-01')
TEST_END      = None   # None = 最新データまで
MODEL_VERSION = 'v2024_2025'

print(f'[学習期間]   {TRAIN_START.date()} ～ {TRAIN_END.date()}')
print(f'[テスト期間] {TEST_START.date()} ～ {"最新" if TEST_END is None else TEST_END.date()}')
print(f'[モデルVer]  {MODEL_VERSION}')
print()
print('※ モックデータ（NB-00）の場合、学習/テスト分割はデータ期間に依存します')
print('   本番データが対象期間を含む場合のみ train/test 分割が有効になります')

## 1. 説明変数の作成

In [ ]:
# --- 学習/テスト期間マスク ---
if 'race_date' not in df.columns:
    raise ValueError('race_date 列が必要です。NB-01 の出力を確認してください。')

df['race_date'] = pd.to_datetime(df['race_date'])

# train: TRAIN_START ≤ race_date ≤ TRAIN_END
# test : TEST_START ≤ race_date (≤ TEST_END if set)
df['is_train'] = (df['race_date'] >= TRAIN_START) & (df['race_date'] <= TRAIN_END)
df['is_test']  = (df['race_date'] >= TEST_START)
if TEST_END is not None:
    df['is_test'] &= (df['race_date'] <= TEST_END)

n_train = df['is_train'].sum()
n_test  = df['is_test'].sum()
n_other = len(df) - n_train - n_test

print(f'データ分割:')
print(f'  学習データ: {n_train:,} 行  ({n_train/len(df)*100:.1f}%)')
print(f'  テストデータ: {n_test:,} 行  ({n_test/len(df)*100:.1f}%)')
print(f'  期間外 (除外): {n_other:,} 行')

if n_train == 0:
    print('\nWARNING: 学習データが 0 件です。TRAIN_START/TRAIN_END を確認してください。')
    print('フォールバック: 全データを学習データとして使用します。')
    df['is_train'] = True

# --- 基準前半スプリット（学習データのみから計算 → データリークを防ぐ）---
df_train = df[df['is_train']].copy()

split_par = (
    df_train.dropna(subset=['split_time_sec'])
    .groupby(['distance', 'surface', 'track_condition'])['split_time_sec']
    .mean()
    .rename('par_split_time_sec')
)
df = df.join(split_par, on=['distance', 'surface', 'track_condition'])

# front_split_dev: 実前半スプリット - 基準スプリット（正 = スロー）
df['front_split_dev'] = df['split_time_sec'] - df['par_split_time_sec']

# --- 馬場補正: TSI_offset（既存 TrackSpeedIndex から取得） ---
if 'tsi_offset' not in df.columns:
    df['tsi_offset'] = 0.0
    print('WARNING: tsi_offset が見つかりません。Δtrack=0 として処理します。')

# --- 斤量補正変数 ---
STD_WEIGHT_MALE   = 55.0
STD_WEIGHT_FEMALE = 53.0

df['std_weight']    = np.where(df['sex'] == '牝', STD_WEIGHT_FEMALE, STD_WEIGHT_MALE)
df['weight_dev']    = df['weight_entry'] - df['std_weight']
df['dist_scale']    = df['distance'] / 2000.0
df['weight_x_dist'] = df['weight_dev'] * df['dist_scale']

# --- レースレベル補正変数（学習データのみから基準値を計算）---
par_log_fq = df.loc[df['is_train'], 'log_fq'].mean()
df['log_fq_dev'] = df['log_fq'] - par_log_fq

print(f'\n基準 log(FQ): {par_log_fq:.4f}  (学習データ平均)')
print('\n=== 説明変数の基本統計（学習データ）===')
print(df.loc[df['is_train'], ['front_split_dev', 'tsi_offset', 'weight_x_dist', 'log_fq_dev']].describe())

In [ ]:
# --- 固定効果ダミー変数 ---
# 距離帯（スプリント/マイル/中距離/長距離）
def distance_band(d):
    if d < 1500:   return 'sprint'
    if d < 1800:   return 'mile'
    if d < 2400:   return 'middle'
    return 'long'

df['distance_band'] = df['distance'].apply(distance_band)

# 固定効果セル = 距離×コース×芝ダート×馬場カテゴリ
# regression formula では C() でカテゴリ変数として扱う
df['fe_cell'] = (
    df['distance'].astype(str) + '_' +
    df['course'] + '_' +
    df['surface'] + '_' +
    df['track_condition']
)

print(f'固定効果セル数: {df["fe_cell"].nunique()}')
print(f'セルあたりの最小サンプル数: {df.groupby("fe_cell").size().min()}')
print(f'30件未満のセル数: {(df.groupby("fe_cell").size() < 30).sum()}')

## 2. OLS 回帰の実行

In [ ]:
# 回帰に使うデータ（必須変数がすべて揃っているレコード）
df_reg = df.dropna(subset=['finish_time_sec', 'fe_cell', 'weight_x_dist', 'log_fq_dev']).copy()

# 前半スプリットが欠損の場合は front_split_dev=0 として回帰に含める
df_reg['front_split_dev'] = df_reg['front_split_dev'].fillna(0)
df_reg['split_available'] = df['split_time_sec'].notna().reindex(df_reg.index).astype(float)

# FQ が欠損の場合は log_fq_dev=0
df_reg['log_fq_dev'] = df_reg['log_fq_dev'].fillna(0)

# 学習/テストフラグを引き継ぐ
df_reg_train = df_reg[df_reg['is_train']].copy()
df_reg_test  = df_reg[df_reg['is_test']].copy()

print(f'回帰使用データ (全体): {len(df_reg):,} 行')
print(f'  学習: {len(df_reg_train):,} 行  テスト: {len(df_reg_test):,} 行')

if len(df_reg_train) == 0:
    raise ValueError('学習データが 0 件です。日付範囲を確認してください。')

# OLS 回帰（学習データのみで係数を推定）
formula = (
    'finish_time_sec ~ '
    'front_split_dev + tsi_offset + weight_x_dist + log_fq_dev '
    '+ C(fe_cell)'
)

model = smf.ols(formula, data=df_reg_train).fit()
print(f'\nOLS 学習完了: R²={model.rsquared:.4f}  Adj.R²={model.rsquared_adj:.4f}')
print(f'学習 RMSE: {np.sqrt(model.mse_resid):.4f} 秒')
print(model.summary())

In [ ]:
# 主要係数の抽出
params  = model.params
pvalues = model.pvalues
conf    = model.conf_int()

key_params = ['front_split_dev', 'tsi_offset', 'weight_x_dist', 'log_fq_dev']
labels     = ['β₁（ペース）', 'β₂（馬場）', 'β₃（斤量）', 'β₄（レースレベル）']

print('=== 主要係数（秒単位）===')
for k, label in zip(key_params, labels):
    if k in params:
        print(f'{label}: coef={params[k]:.4f}  p={pvalues[k]:.4f}'
              f'  95%CI=[{conf.loc[k,0]:.4f}, {conf.loc[k,1]:.4f}]')

# --- 学習 RMSE ---
train_rmse = np.sqrt(model.mse_resid)

# --- テスト RMSE（テストデータに学習済みモデルを適用）---
if len(df_reg_test) > 0:
    test_pred = model.predict(df_reg_test)
    test_resid = df_reg_test['finish_time_sec'] - test_pred
    test_rmse = np.sqrt((test_resid ** 2).mean())
    overfit_ratio = test_rmse / train_rmse
    print(f'\n=== RMSE 学習/テスト比較 ===')
    print(f'  学習 RMSE: {train_rmse:.4f} 秒')
    print(f'  テスト RMSE: {test_rmse:.4f} 秒')
    print(f'  過学習比率 (test/train): {overfit_ratio:.3f}  (1.0 = 過学習なし)')
else:
    test_rmse = None
    print(f'\nR²: {model.rsquared:.4f}')
    print(f'学習 RMSE: {train_rmse:.4f} 秒')
    print('テストデータなし（期間外）')

## 3. 残差分析

In [ ]:
# 残差分析（学習データのみ）
df_reg_train['residual'] = model.resid
df_reg_train['fitted']   = model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(df_reg_train['residual'], bins=80, color='steelblue', alpha=0.8)
axes[0].set_title(f'残差分布（学習データ, n={len(df_reg_train):,}）')
axes[0].set_xlabel('残差（秒）')

axes[1].scatter(df_reg_train['fitted'], df_reg_train['residual'], alpha=0.1, s=2)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('フィッテッド値')
axes[1].set_ylabel('残差')
axes[1].set_title('フィッテッド vs 残差（学習）')

sm.qqplot(df_reg_train['residual'], line='s', ax=axes[2])
axes[2].set_title('QQ プロット（学習残差）')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'residual_analysis.png', dpi=120)
plt.show()

print(f'学習残差 std: {df_reg_train["residual"].std():.3f} 秒')
print(f'1点=0.1秒 換算での残差 SD: {df_reg_train["residual"].std()*10:.1f} ポイント')

## 4. めぐ指数の計算と基準タイムの保存

In [ ]:
# 確定した係数
beta_pace   = params.get('front_split_dev', 0)
beta_track  = params.get('tsi_offset', 0)
beta_weight = params.get('weight_x_dist', 0)
beta_level  = params.get('log_fq_dev', 0)

# === 学習データのめぐ指数（fitted values を使用）===
df_reg_train = df_reg_train.copy()
df_reg_train['fitted']   = model.fittedvalues

# === テストデータのめぐ指数（predict を使用）===
if len(df_reg_test) > 0:
    df_reg_test = df_reg_test.copy()
    # 学習セルに存在しない fe_cell は statsmodels が 0 係数として扱う（intercept のみ）
    df_reg_test['fitted'] = model.predict(df_reg_test)

# 全データを結合して一括処理
df_all = pd.concat([df_reg_train, df_reg_test], ignore_index=False).sort_index()

# 各補正量の計算
# beta_track は OLS 推定値（負値: 速い馬場 → タイム短）
# delta_track = beta_track × tsi_offset（符号反転なし）
df_all['delta_pace']   = beta_pace   * df_all['front_split_dev']
df_all['delta_track']  = beta_track  * df_all['tsi_offset']
df_all['delta_weight'] = beta_weight * df_all['weight_x_dist']
df_all['delta_level']  = beta_level  * df_all['log_fq_dev']

# 補正済みタイム
df_all['adjusted_time'] = (
    df_all['finish_time_sec']
    - df_all['delta_pace']
    - df_all['delta_track']
    - df_all['delta_weight']
    - df_all['delta_level']
)

# 基準タイム = fitted - 各補正変数の寄与をすべて除く
df_all['par_time'] = (
    df_all['fitted']
    - beta_pace   * df_all['front_split_dev']
    - beta_track  * df_all['tsi_offset']
    - beta_weight * df_all['weight_x_dist']
    - beta_level  * df_all['log_fq_dev']
)

# めぐ指数: megu_index = 100 - residual × 10 (by construction)
df_all['megu_index'] = 100 + (df_all['par_time'] - df_all['adjusted_time']) * 10

# split ラベル
df_all['split'] = 'other'
df_all.loc[df_all['is_train'], 'split'] = 'train'
df_all.loc[df_all['is_test'],  'split'] = 'test'

print('=== めぐ指数の分布（学習/テスト別）===')
print(df_all.groupby('split')['megu_index'].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for split, color in [('train', 'steelblue'), ('test', 'coral')]:
    sub = df_all[df_all['split'] == split]
    if len(sub) > 0:
        axes[0].hist(sub['megu_index'], bins=80, color=color, alpha=0.6, label=f'{split} (n={len(sub):,})')
axes[0].axvline(100, color='red', linestyle='--', label='par=100')
axes[0].set_xlabel('めぐ指数')
axes[0].set_title('めぐ指数の分布（学習/テスト）')
axes[0].legend()

df_all.groupby(['split', 'surface'])['megu_index'].mean().unstack().plot(
    kind='bar', ax=axes[1], color=['steelblue', 'coral', 'gray']
)
axes[1].set_title('surface × split 別平均めぐ指数')
axes[1].set_xlabel('')
axes[1].legend(title='split')
axes[1].axhline(100, color='red', linestyle='--')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'megu_index_distribution.png', dpi=120)
plt.show()

In [ ]:
# 基準タイムを距離×コース×芝ダート×馬場カテゴリ別に集計
# 学習データのみを使用: テストデータで見た新セルの基準タイムが汚染されないよう
df_train_only = df_all[df_all['split'] == 'train']

par_time_table = (
    df_train_only.groupby(['distance', 'course', 'surface', 'track_condition'])
    .agg(
        par_time_sec=('par_time', 'mean'),
        par_front_split_sec=('par_split_time_sec', 'mean'),
        sample_count=('finish_time_sec', 'count')
    )
    .reset_index()
)
par_time_table['model_version'] = MODEL_VERSION

print(f'基準タイムセル数: {len(par_time_table):,}')
print(par_time_table.head(10))
par_time_table.to_parquet(OUTPUT_DIR / 'megu_par_time.parquet', index=False)

In [ ]:
# DB への保存（MODEL_VERSION は冒頭の設定セルから参照）
from src.db.session import get_session, init_engine
from sqlalchemy.dialects.postgresql import insert as pg_insert
from src.db.models import MeguRegressionParams, MeguParTime
import datetime

init_engine()

param_rows = [
    {'param_name': 'beta_pace',   'param_value': float(beta_pace),   'model_version': MODEL_VERSION},
    {'param_name': 'beta_track',  'param_value': float(beta_track),  'model_version': MODEL_VERSION},
    {'param_name': 'beta_weight', 'param_value': float(beta_weight), 'model_version': MODEL_VERSION},
    {'param_name': 'beta_level',  'param_value': float(beta_level),  'model_version': MODEL_VERSION},
    {'param_name': 'par_log_fq',  'param_value': float(par_log_fq),  'model_version': MODEL_VERSION},
    {'param_name': 'train_start', 'param_value': float(TRAIN_START.timestamp()), 'model_version': MODEL_VERSION},
    {'param_name': 'train_end',   'param_value': float(TRAIN_END.timestamp()),   'model_version': MODEL_VERSION},
]

with get_session() as session:
    stmt = pg_insert(MeguRegressionParams).values(param_rows)
    stmt = stmt.on_conflict_do_update(
        constraint='megu_regression_params_param_name_model_version_key',
        set_={'param_value': stmt.excluded.param_value}
    )
    session.execute(stmt)
    session.commit()

print(f'回帰係数 {len(param_rows)} 件を DB に保存しました (model_version={MODEL_VERSION})')

In [ ]:
# めぐ指数結果を保存（NB-05/06 で使用）
# split 列: 'train' | 'test' | 'other' → NB-05はtrainでチューニング、NB-06はtestで評価
save_cols = [
    'race_id', 'horse_id', 'race_date', 'split',
    'finish_time_sec', 'par_time',
    'delta_pace', 'delta_track', 'delta_weight', 'delta_level',
    'adjusted_time', 'megu_index',
]
save_cols_available = [c for c in save_cols if c in df_all.columns]
df_all[save_cols_available].to_parquet(OUTPUT_DIR / 'megu_index_results.parquet', index=False)
print(f'めぐ指数計算結果を保存しました (split 列付き)')
print(f'  保存行数: {len(df_all):,}  保存列: {save_cols_available}')

# 係数サマリー
print(f'\n=== 確定係数サマリー (model_version={MODEL_VERSION}) ===')
print(f'β₁（ペース補正）  : {beta_pace:.4f} 秒/秒  → 前半1秒遅いと全体で {beta_pace:.3f}秒補正')
print(f'β₂（馬場補正）    : {beta_track:.4f} 秒  → TSI 1秒速い日で {beta_track:.3f}秒補正')
print(f'β₃（斤量補正）    : {beta_weight:.4f} 秒/kg  → 2000mで1kg重いと {beta_weight:.3f}秒補正 (理論値: 0.2)')
print(f'β₄（レースレベル）: {beta_level:.4f} 秒/log → FQ が2倍になると {beta_level*np.log(2):.3f}秒補正')

# 学習/テスト分布の比較
print(f'\n=== 学習/テスト期間別めぐ指数 ===')
summary = df_all.groupby('split')['megu_index'].agg(['count', 'mean', 'std']).round(3)
summary.columns = ['件数', '平均', '標準偏差']
print(summary)

if test_rmse is not None:
    print(f'\n=== 汎化性能 ===')
    print(f'  学習 RMSE: {train_rmse:.4f} 秒')
    print(f'  テスト RMSE: {test_rmse:.4f} 秒')
    print(f'  過学習比率: {test_rmse/train_rmse:.3f}')